In [249]:
# This Data pipeline is created to extract, read, transform, write Lagos weather csv data into a DataBase
# The first stage of our Pipeline is Reading and Extracting the CSV using Pandas Dataframe.
# The second stage will be Staging the Data
# The third stage will be to Vaidate the Data
# The fouth stage will be to Clean and Transform the Data.
# The fifth stage is UPSERT which loads the transformed/cleaned data into a new table
# The sixth stage is to Log Errors/Success of our Lagos weather data
# The final stage is to write the Data into a Data Warehouse for analysis.

In [250]:
import pandas as pd

In [251]:
lagos_df = pd.read_csv("Lagos_weather.csv", skiprows=3)

In [252]:
# After extracting and reading the csv data as from above, we show it to be sure it is extracted.
print(lagos_df.head(20))

          time  temperature_2m_max (°C)  temperature_2m_min (°C)  \
0   2020-01-01                     33.4                     21.6   
1   2020-01-02                     33.1                     21.9   
2   2020-01-03                     33.3                     21.6   
3   2020-01-04                     33.3                     20.9   
4   2020-01-05                     32.1                     20.9   
5   2020-01-06                     32.8                     21.8   
6   2020-01-07                     33.0                     22.2   
7   2020-01-08                     32.7                     23.7   
8   2020-01-09                     32.3                     23.1   
9   2020-01-10                     32.2                     25.9   
10  2020-01-11                     30.5                     26.2   
11  2020-01-12                     31.1                     24.7   
12  2020-01-13                     31.4                     25.6   
13  2020-01-14                     33.3         

In [253]:
lagos_df.shape # to know the nuber of rows and column

(1827, 4)

In [254]:
lagos_df.describe()

,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm)
count,1827.000000,1827.000000,1827.000000
mean,30.174767,24.706787,3.501478
std,2.049252,1.325931,7.307314
min,25.300000,18.700000,0.000000
25%,28.400000,23.900000,0.300000
50%,30.300000,24.600000,1.200000
75%,31.900000,25.500000,3.600000
max,34.900000,28.600000,148.500000


In [255]:
lagos_df.dtypes

time                        object
temperature_2m_max (°C)    float64
temperature_2m_min (°C)    float64
precipitation_sum (mm)     float64
dtype: object

In [256]:
# The next (Second) phase is "STAGING", where we save/keep/store the raw data in SQLite before making any changes to it in the Transformation phase, 
# incase of a bug we can refer back to our raw data.

In [257]:
# Connecting to SQLite to create an engine.

from sqlalchemy import create_engine

engine = create_engine("sqlite:///lag_weather.db")

In [258]:
# Let's build the Raw Staging Copy with these blocks of code

stg_df = lagos_df.copy() # We make a Copy of the raw untouched data
stg_df["source_file"] = "Lagos_weather.csv" # We record which files the rows came from
stg_df["loaded_at"] = pd.Timestamp.now().isoformat() # We record when they were loaded.
stg_df.head() # We check what we have done

,time,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),source_file,loaded_at
0,2020-01-01,33.4,21.6,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
1,2020-01-02,33.1,21.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
2,2020-01-03,33.3,21.6,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
3,2020-01-04,33.3,20.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
4,2020-01-05,32.1,20.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675


In [259]:
# Now, let's write the data into the SQLite database

stg_df.to_sql("stg_weather", engine, if_exists="replace", index=False)

1827

In [260]:
# Let's confirm if it worked
pd.read_sql("SELECT * FROM stg_weather LIMIT 5", engine)

,time,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),source_file,loaded_at
0,2020-01-01,33.4,21.6,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
1,2020-01-02,33.1,21.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
2,2020-01-03,33.3,21.6,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
3,2020-01-04,33.3,20.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675
4,2020-01-05,32.1,20.9,0.0,Lagos_weather.csv,2026-10-23T12:12:32.402675


In [261]:
# Let's Confirm the table holds every row. 

pd.read_sql("SELECT COUNT(*) AS total_rows FROM stg_weather", engine)

,total_rows
0,1827


In [262]:
# The Next(Third) Phase is Validation of our data, where we check the accuracy or authenticity of our data.
# For the Validation we are checking the list as follows below:
# 1. Schema Check : to confirm our file has the four columns we expect
# 2. Completeness Check : row count, null values, no missing days.
# 3. Uniqueness : No duplicate days.
# 4. Plausibility : This checks that the values are realistic for Lagos. 
# A number can be present and unique but still wrong, such as a temperature of 400 or negative rainfall.
# 5. Consistency Check : This checks that values agree with each other. 
# On any given day, the maximum temperature can't be lower than the minimum temperature.
# 6. Timelessness Check: This checks that the data isn't from the future, and shows how recent it is. 
# Future dates usually mean a bad date format or a bug at the source

In [263]:
# 1. Schema Check for number of columns
expected_columns = [
    "time",
    "temperature_2m_max (°C)",
    "temperature_2m_min (°C)",
    "precipitation_sum (mm)",
]
list(lagos_df.columns) == expected_columns

True

In [264]:
# 2. Completeness check
# For row count we do:
len(lagos_df) > 0 # If it's true it means we are not loading an empty dataset to our database. 
# If it was false it means we loaded empty datasets and our pipeline will still run.

True

In [265]:
len(lagos_df)

1827

In [266]:
# for null values we do :
lagos_df.isnull().sum() # If it's 0 that means there are no null values, if it's more than 0 that means there are Null values.

time                       0
temperature_2m_max (°C)    0
temperature_2m_min (°C)    0
precipitation_sum (mm)     0
dtype: int64

In [267]:
# We check for no missing days
dates = pd.to_datetime(lagos_df["time"]) # converts the text dates to real dates for this check only.
expected_days = pd.date_range(dates.min(), dates.max(), freq="D") # builds every day from your first date to your last. "D" means daily.

len(expected_days) == len(lagos_df)

True

In [268]:
# 3. Uniqueness Check. 
#No date should appear twice, it will affect our rows and aggregation later.
lagos_df["time"].is_unique

# If it runs false this code below shows the duplicated days
# lagos_df[lagos_df["time"].duplicated(keep=False)]

True

In [269]:
# 4. Plausibility
# First look at the minimum and maximun values
lagos_df.describe()

,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm)
count,1827.000000,1827.000000,1827.000000
mean,30.174767,24.706787,3.501478
std,2.049252,1.325931,7.307314
min,25.300000,18.700000,0.000000
25%,28.400000,23.900000,0.300000
50%,30.300000,24.600000,1.200000
75%,31.900000,25.500000,3.600000
max,34.900000,28.600000,148.500000


In [270]:
# Then let's run the false/true checks

temp_max_ok = lagos_df["temperature_2m_max (°C)"].between(15, 45).all()
temp_min_ok = lagos_df["temperature_2m_min (°C)"].between(15, 45).all()
rain_ok = (lagos_df["precipitation_sum (mm)"] >= 0).all()

temp_max_ok, temp_min_ok, rain_ok

(np.True_, np.True_, np.True_)

In [271]:
# 5. Consistency Check
# The maximum Temperature must always be greater than the minimum in a day
(lagos_df["temperature_2m_max (°C)"] >= lagos_df["temperature_2m_min (°C)"]).all()

np.True_

In [272]:
# 6. Timelessness Check.
dates = pd.to_datetime(lagos_df["time"])
dates.max() <= pd.Timestamp.today()

True

In [273]:
# THE VALIDTION CHECK IS OVER
# NEXT WE BUNDLE ALL THE CHECKS INTO ONE FUNCTION

In [274]:
def validate_weather(df):
    dates = pd.to_datetime(df["time"], errors="coerce")

    checks = {
        "schema": list(df.columns) == expected_columns,
        "row_count": len(df) > 0,
        "no_nulls": not df.isnull().any().any(),
        "valid_dates": dates.notnull().all(),
        "unique_dates": df["time"].is_unique,
        "no_missing_days": len(pd.date_range(dates.min(), dates.max(), freq="D")) == len(df),
        "temp_max_range": df["temperature_2m_max (°C)"].between(15, 45).all(),
        "temp_min_range": df["temperature_2m_min (°C)"].between(15, 45).all(),
        "rain_range": (df["precipitation_sum (mm)"] >= 0).all(),
        "max_gte_min": (df["temperature_2m_max (°C)"] >= df["temperature_2m_min (°C)"]).all(),
        "no_future_dates": dates.max() <= pd.Timestamp.today(),
    }
    return {name: bool(result) for name, result in checks.items()}

In [275]:
results = validate_weather(lagos_df)
results

{'schema': True,
 'row_count': True,
 'no_nulls': True,
 'valid_dates': True,
 'unique_dates': True,
 'no_missing_days': True,
 'temp_max_range': True,
 'temp_min_range': True,
 'rain_range': True,
 'max_gte_min': True,
 'no_future_dates': True}

In [276]:
all(results.values()) # is True only if every check passed. That's the signal the pipeline will use to decide btw continuing and rejecting the file.

True

In [277]:
all(validate_weather(lagos_df).values())

True

In [278]:
# WE ARE IN THE FOURTH STAGE WHICH IS THE TRANSFORMATION.

In [279]:
# Transform makes the data clean and SQL-friendly. First, let's work on a copy so our original lagos_df stays untouched:

clean_lagos_df = lagos_df.copy()

In [280]:
# Lets rename the columns
clean_lagos_df = clean_lagos_df.rename(columns={
    "time" : "date",
    "temperature_2m_max (°C)" : "temp_max_c",
    "temperature_2m_min (°C)" : "temp_min_c",
    "precipitation_sum (mm)" : "precip_mm"
})
clean_lagos_df.head()

,date,temp_max_c,temp_min_c,precip_mm
0,2020-01-01,33.4,21.6,0.0
1,2020-01-02,33.1,21.9,0.0
2,2020-01-03,33.3,21.6,0.0
3,2020-01-04,33.3,20.9,0.0
4,2020-01-05,32.1,20.9,0.0


In [281]:
# convert date to a real date
clean_lagos_df["date"] = pd.to_datetime(clean_lagos_df["date"])

In [282]:
# Let's check out conveersion
clean_lagos_df.dtypes

date          datetime64[ns]
temp_max_c           float64
temp_min_c           float64
precip_mm            float64
dtype: object

In [283]:
# Let's add a city column to indicate location since the data came without a location column.
clean_lagos_df["city"] = "Lagos"

In [284]:
# Let's move it to the front of the table
clean_lagos_df = clean_lagos_df[["city", "date", "temp_max_c", "temp_min_c", "precip_mm"]]

clean_lagos_df.head()

,city,date,temp_max_c,temp_min_c,precip_mm
0,Lagos,2020-01-01,33.4,21.6,0.0
1,Lagos,2020-01-02,33.1,21.9,0.0
2,Lagos,2020-01-03,33.3,21.6,0.0
3,Lagos,2020-01-04,33.3,20.9,0.0
4,Lagos,2020-01-05,32.1,20.9,0.0


In [285]:
clean_lagos_df.dtypes

city                  object
date          datetime64[ns]
temp_max_c           float64
temp_min_c           float64
precip_mm            float64
dtype: object

In [286]:
# drop duplicates (safety net)
# Your uniqueness check already passed, so this should remove nothing from today’s file. 
# It’s there to protect the pipeline when a future file does contain repeats

rows_before = len(clean_lagos_df)
clean_lagos_df = clean_lagos_df.drop_duplicates(subset=["city", "date"]) # defines what “repeated” means: two rows are duplicates/ 
# when they have the same city and the same date
rows_after = len(clean_lagos_df)
rows_before, rows_after

(1827, 1827)

In [287]:
# The next phase is the UPSERT which is to load the cleaned data into the clean table

In [288]:
# Let's create the new table right away
from sqlalchemy import text

create_sql = """
CREATE TABLE IF NOT EXISTS weather_daily (
    city TEXT NOT NULL,
    date TEXT NOT NULL,
    temp_max_c REAL,
    temp_min_c REAL,
    precip_mm REAL,
    PRIMARY KEY (city, date)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_sql))

In [289]:
# Let's check our new created table if it happened.
pd.read_sql("PRAGMA table_info(weather_daily)", engine)

,cid,name,type,notnull,dflt_value,pk
0,0,city,TEXT,1,None,1
1,1,date,TEXT,1,None,2
2,2,temp_max_c,REAL,0,None,0
3,3,temp_min_c,REAL,0,None,0
4,4,precip_mm,REAL,0,None,0


In [290]:
# Now let's load our clean_lagos_df into the new table

upsert_sql = """
INSERT INTO weather_daily (city, date, temp_max_c, temp_min_c, precip_mm)
VALUES (:city, :date, :temp_max_c, :temp_min_c, :precip_mm)
ON CONFLICT (city, date) DO UPDATE SET
    temp_max_c = excluded.temp_max_c,
    temp_min_c = excluded.temp_min_c,
    precip_mm  = excluded.precip_mm;
"""

load_df = clean_lagos_df.copy()
load_df["date"] = load_df["date"].dt.strftime("%Y-%m-%d")

records = load_df.to_dict(orient="records")

with engine.begin() as conn:
    conn.execute(text(upsert_sql), records)

In [291]:
# Let's check what we just loaded if it worked
pd.read_sql("SELECT COUNT(*) AS total_rows FROM weather_daily", engine)

,total_rows
0,1827


In [292]:
len(clean_lagos_df)

1827

In [293]:
# THE NEXT STAGE IS THE LOGGING PHASE WHICH IS TO RECORD WHAT HAPPENS ON EACH RUN.

In [294]:
# Let's create the log table that records our runs

log_sql = """
CREATE TABLE IF NOT EXISTS etl_run_log (
    run_id INTEGER PRIMARY KEY AUTOINCREMENT,
    file_name TEXT NOT NULL,
    file_hash TEXT,
    rows_read INTEGER,
    rows_loaded INTEGER,
    status TEXT NOT NULL,
    error_message TEXT,
    run_at TEXT NOT NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(log_sql))

In [295]:
# Let's check if it worked
pd.read_sql("PRAGMA table_info(etl_run_log)", engine)

,cid,name,type,notnull,dflt_value,pk
0,0,run_id,INTEGER,0,None,1
1,1,file_name,TEXT,1,None,0
2,2,file_hash,TEXT,0,None,0
3,3,rows_read,INTEGER,0,None,0
4,4,rows_loaded,INTEGER,0,None,0
5,5,status,TEXT,1,None,0
6,6,error_message,TEXT,0,None,0
7,7,run_at,TEXT,1,None,0


In [296]:
# let's write our first run into the log

from src.extract import file_hash, get_processed_hashes
from src.load import log_run

file_name = "Lagos_weather.csv"
h = file_hash(file_name)

if h in get_processed_hashes(engine):
    print("Already logged, skipping")
else:
    log_run(engine, file_name, h, len(lagos_df), len(clean_lagos_df), "success")
    print("Logged")

Already logged, skipping


In [297]:
# let's check the result

pd.read_sql("SELECT * FROM etl_run_log", engine)

,run_id,file_name,file_hash,rows_read,rows_loaded,status,error_message,run_at
0,1,Lagos_weather.csv,dd4fa16ff692468b2cf263ae71cd06227abd797dc224ef...,1827,1827,success,None,2026-10-19T16:46:05.695236
1,5,Lagos_weather_test.csv,1df0bcf6be3474f551534bebc602c84dd564c604acc8dd...,1827,1827,success,None,2026-10-23T11:02:30.771424
2,7,Lagos_broken.csv,cc9f071999f1878ca8d3325a06c8bef9cb692283cd0048...,1827,0,rejected,Failed checks: temp_max_range,2026-10-23T11:55:05.507825


In [298]:
# NOW WE ARE DONE WITH THE WHOLE TRANSFORMATION PROCESS, LETS MOVE THE FILE TO ANOTHER FOLDER.

In [299]:
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/rejected").mkdir(parents=True, exist_ok=True)

# This makes two folders called processed and rejected inside a folder called data. 
# If they already exist, nothing happens. We will check your project folder afterwards and we will should see them.

In [300]:
# put a copy of your CSV in the processed folder.

import shutil

shutil.copy("Lagos_weather.csv", "data/processed/Lagos_weather.csv")

'data/processed/Lagos_weather.csv'

In [301]:
# let's check if it worked
from pathlib import Path

list(Path("data/processed").glob("*.csv"))

[WindowsPath('data/processed/Lagos_weather.csv'),
 WindowsPath('data/processed/Lagos_weather_test.csv')]

In [302]:
# The next session is to run a schedular using .py script because our schedular cannot run on notebook cells.
# So i have created a src folder on VScode and i have extracted the application over there.
# Now, let us test if we've done the correct thing.

from src.extract import read_weather_csv

test_df = read_weather_csv("Lagos_weather.csv")
test_df.shape

(1827, 4)

In [303]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [304]:
from src.extract import file_hash

file_hash("Lagos_weather.csv")

'dd4fa16ff692468b2cf263ae71cd06227abd797dc224ef3a6bdfb90c1c0ceeeb'

In [305]:
# Let's compare the hash_file on the VScode is same as on our log on notebook

log_hashes = pd.read_sql("SELECT file_hash FROM etl_run_log", engine)["file_hash"].tolist()

file_hash("Lagos_weather.csv") in log_hashes

True

In [306]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [307]:
# Let's test the fingerprints of successful runs from etl_run_log into a set
# is_new_file returns True for a new file and False for one the pipeline has seen
from src.extract import get_processed_hashes, is_new_file

hashes = get_processed_hashes(engine)
is_new_file("Lagos_weather.csv", hashes)

False

In [308]:
# It return False, because Lagos_weather.csv is already in the log as a success, so it’s not new

In [309]:
# Now, we will check our inbox folder where new CSVs are dropped, then find_new_files returns a list of the new files, 
# sorted by name, that have not been processed yet

# let's create the inbox folder and put a copy of our CSV in it.
from pathlib import Path
import shutil

Path("data/inbox").mkdir(parents=True, exist_ok=True)
shutil.copy("Lagos_weather.csv", "data/inbox/Lagos_weather.csv")

'data/inbox/Lagos_weather.csv'

In [310]:
# Now, let's run this function to test it
from src.extract import find_new_files

find_new_files("data/inbox", hashes)

[]

In [311]:
# It returns an empty list, []. The file is in the inbox, but its hash is already in the log, so the pipeline treats it as seen and skips it.

In [312]:
# Okay now, just for clarification purpose, let's prove our pipeline detects a new file
text_data = Path("Lagos_weather.csv").read_text()
Path("data/inbox/Lagos_weather_test.csv").write_text(text_data + "\n")
# this is a changed copy of the file in the inbox, with a different name and one extra blank line at the end so the contents differ.
find_new_files("data/inbox", hashes)

[]

In [313]:
# Apparently find_new_files detects another file that has not been processed yet.

In [314]:
# Let's clean it up
from pathlib import Path

Path("data/inbox/Lagos_weather_test.csv").unlink()

In [315]:
# Confirmation that it's gone
list(Path("data/inbox").glob("*.csv"))

[WindowsPath('data/inbox/Lagos_weather.csv')]

In [316]:
# OUR NEXT SESSION ON VScode IS TO CREATE A VALIDATE.PY FOR OUR DATA VALIDATION

In [317]:
# We've created, now let's test it
from src.validate import validate_weather

results = validate_weather(lagos_df)
all(results.values())

True

In [318]:
# OUR NEXT SESSION ON VScode IS TO CREATE A TRANFORM.PY FOR OUR DATA TRANSFORMATION

In [319]:
# We've created a transform.py, noe let's test it.
from src.transform import transform_weather

test_clean = transform_weather(lagos_df, "Lagos")
test_clean.head()

,city,date,temp_max_c,temp_min_c,precip_mm
0,Lagos,2020-01-01,33.4,21.6,0.0
1,Lagos,2020-01-02,33.1,21.9,0.0
2,Lagos,2020-01-03,33.3,21.6,0.0
3,Lagos,2020-01-04,33.3,20.9,0.0
4,Lagos,2020-01-05,32.1,20.9,0.0


In [320]:
test_clean.equals(clean_lagos_df)

True

In [321]:
# OUR NEXT SESSION IS TO LOAD.PY WHICH COVERS STAGING AND UPSERT THAT WRITES TO A DATABASE AFTER TRANSFORMATION.

In [322]:
# Let's test it and see how far
from sqlalchemy import create_engine
from src.load import create_tables, load_staging, upsert_weather

test_engine = create_engine("sqlite:///:memory:")

create_tables(test_engine)
load_staging(lagos_df, test_engine, "Lagos_weather.csv")
upsert_weather(test_clean, test_engine)
upsert_weather(test_clean, test_engine)   # run it twice on purpose

1827

In [323]:
# Let's count the rows in both tables
pd.read_sql("SELECT COUNT(*) AS rows_in_weather_daily FROM weather_daily", test_engine)
pd.read_sql("SELECT COUNT(*) AS rows_in_staging FROM stg_weather", test_engine)

,rows_in_staging
0,1827


In [324]:
# Even though the upsert ran twice, weather_daily should not double, which proves the upsert is still idempotent inside the script.

In [325]:
# Let's test tun our LOGGING (success/error)
# Test it against the temporary database
# Use the in-memory test_engine again so your real log stays clean. 
# If your notebook was restarted, run the test_engine setup cell from the last step first.

from src.load import log_run
from src.extract import file_hash, get_processed_hashes

h = file_hash("Lagos_weather.csv")

log_run(test_engine, "Lagos_weather.csv", h, 1827, 1827, "success")
log_run(test_engine, "Bad_file.csv", "fakehash123", 1827, 0, "rejected", "temp_max_range failed")

pd.read_sql("SELECT * FROM etl_run_log", test_engine)

,run_id,file_name,file_hash,rows_read,rows_loaded,status,error_message,run_at
0,1,Lagos_weather.csv,dd4fa16ff692468b2cf263ae71cd06227abd797dc224ef...,1827,1827,success,None,2026-10-23T12:12:43.601027
1,2,Bad_file.csv,fakehash123,1827,0,rejected,temp_max_range failed,2026-10-23T12:12:43.601977


In [326]:
get_processed_hashes(test_engine)

{'dd4fa16ff692468b2cf263ae71cd06227abd797dc224ef3a6bdfb90c1c0ceeeb'}

In [327]:
# The set contained your real file’s fingerprint, so get_processed_hashes is doing its job. 
# One quick check confirms the other half, that the rejected file is not in the set:

"fakehash123" in get_processed_hashes(test_engine)

False

In [328]:
# THE NEXT SESSION: the Move file function
# In the notebook we copied the file by hand. 
# The pipeline needs a function that files it away after each run: processed/ if it succeeded, rejected/ if it failed.

In [329]:
# we will test with a dummy file
from pathlib import Path

Path("data/inbox").mkdir(parents=True, exist_ok=True)
Path("data/inbox/move_test.csv").write_text("test")

4

In [330]:
# Let's move it
from src.files import move_file

move_file("data/inbox/move_test.csv", "success")

WindowsPath('data/processed/move_test.csv')

In [331]:
# Check where it went
list(Path("data/inbox").glob("move_test.csv")), list(Path("data/processed").glob("move_test.csv"))

([], [WindowsPath('data/processed/move_test.csv')])

In [332]:
# Let's clean up
Path("data/processed/move_test.csv").unlink()

In [333]:
import sys
sys.executable

'C:\\ProgramData\\anaconda3\\python.exe'

In [334]:
# Let's feed our pipeline a new file and see if it will detect a new file and run a file it has not see before.
# We’ve tested the stages one by one in the notebook. This is the first time they all run together, automatically, from one script

In [335]:
from pathlib import Path

text_data = Path("Lagos_weather.csv").read_text()
Path("data/inbox/Lagos_weather_test.csv").write_text(text_data + "\n")

47875

In [336]:
# Let's go and run "& C:\\ProgramData\\anaconda3\\python.exe run_pipeline.py" on our VScode terminal to confirm the file ran.

In [337]:
# Let's check if the log recorded it
pd.read_sql("SELECT file_name, rows_read, rows_loaded, status FROM etl_run_log", engine)

,file_name,rows_read,rows_loaded,status
0,Lagos_weather.csv,1827,1827,success
1,Lagos_weather_test.csv,1827,1827,success
2,Lagos_broken.csv,1827,0,rejected


In [338]:
# log above is supposed to be just 2 rows, ! lagos_weather.csv and 1 lagos_weather_test.csv. 
# But because i ran an ALL-Rerun cells multiple times, hence the multiple lagos_weather.csv rows.

pd.read_sql("""
    SELECT run_id, file_name, substr(file_hash, 1, 8) AS hash_start, status, run_at
    FROM etl_run_log
    ORDER BY run_id
""", engine)
# We are checking the file_hash fingerprint to be sure they are duplicates and not fresh files apart form the ...test.csv file.

,run_id,file_name,hash_start,status,run_at
0,1,Lagos_weather.csv,dd4fa16f,success,2026-10-19T16:46:05.695236
1,5,Lagos_weather_test.csv,1df0bcf6,success,2026-10-23T11:02:30.771424
2,7,Lagos_broken.csv,cc9f0719,rejected,2026-10-23T11:55:05.507825


In [339]:
# Let's stop an delete the duplicates

In [340]:
#1. Delete the manual log-insert cell (the one with `log_row` and `insert_log_sql`), or comment out its last two lines (`with engine.begin()...` and `conn.execute(...)`).
#2. Never write to `etl_run_log` from the notebook again. Let `run_pipeline.py` be the only thing that writes to it.
#3. Run the delete-duplicates cell once, so the log keeps one row per file.
#4. Before any Run All, scan for cells that write to the database (`to_sql`, `INSERT`, `conn.execute`), and comment them out or move them to scripts.

In [341]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        DELETE FROM etl_run_log
        WHERE run_id NOT IN (
            SELECT MIN(run_id) FROM etl_run_log GROUP BY file_hash
        )
    """))

In [342]:
# Let's check if it worked

pd.read_sql("""
    SELECT run_id, file_name, substr(file_hash, 1, 8) AS hash_start, status
    FROM etl_run_log
    ORDER BY run_id
""", engine)

,run_id,file_name,hash_start,status
0,1,Lagos_weather.csv,dd4fa16f,success
1,5,Lagos_weather_test.csv,1df0bcf6,success
2,7,Lagos_broken.csv,cc9f0719,rejected


In [343]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM weather_daily", engine)

,total_rows
0,1827


In [344]:
# Let's check the “fail” branch of our DAG. So far we’ve only seen the pipeline accept files, 
# so now we’ll give it a deliberately broken file and confirm it rejects it and protects your clean table

In [345]:
# Let's make a broken file
from pathlib import Path

lines = Path("Lagos_weather.csv").read_text().splitlines()
print(lines[4])   # shows the first data row, before the change

2020-01-01,33.4,21.6,0.00


In [346]:
lines[4] = "2020-01-01,99.0,21.6,0.0"   # 99 °C max temperature is impossible for Lagos
Path("data/inbox/Lagos_broken.csv").write_text("\n".join(lines) + "\n")

47873

In [347]:
# Let's check the result, was it logged in as rejected and with reason?
pd.read_sql("SELECT file_name, rows_loaded, status, error_message FROM etl_run_log", engine)

,file_name,rows_loaded,status,error_message
0,Lagos_weather.csv,1827,success,None
1,Lagos_weather_test.csv,1827,success,None
2,Lagos_broken.csv,0,rejected,Failed checks: temp_max_range


In [348]:
# Is the clean table untouched
pd.read_sql("SELECT COUNT(*) AS total_rows FROM weather_daily", engine)

,total_rows
0,1827


In [349]:
pd.read_sql("SELECT MAX(temp_max_c) AS highest FROM weather_daily", engine)

,highest
0,34.9


In [350]:
# Was the file moved into the correct folder?
list(Path("data/rejected").glob("*.csv")), list(Path("data/inbox").glob("*.csv"))

([WindowsPath('data/rejected/Lagos_broken.csv')],
 [WindowsPath('data/inbox/Lagos_broken.csv'),
  WindowsPath('data/inbox/Lagos_weather.csv'),
  WindowsPath('data/inbox/Lagos_weather_test.csv')])

In [351]:
# Let's delete all our test files because we are done with them.
# Preview
from pathlib import Path

to_delete = (
    list(Path("data/inbox").glob("*.csv"))                    # everything left in the inbox
    + list(Path("data/rejected").glob("Lagos_broken.csv"))    # test file
    + list(Path("data/processed").glob("Lagos_weather_test.csv"))  # test file
    + list(Path("data/processed").glob("move_test.csv"))      # test file
)

for p in to_delete:
    print(p)

data\inbox\Lagos_broken.csv
data\inbox\Lagos_weather.csv
data\inbox\Lagos_weather_test.csv
data\rejected\Lagos_broken.csv
data\processed\Lagos_weather_test.csv


In [352]:
# let's delete
for p in to_delete:
    p.unlink(missing_ok=True)

print("Inbox:", list(Path("data/inbox").glob("*.csv")))
print("Rejected:", list(Path("data/rejected").glob("*.csv")))
print("Processed:", list(Path("data/processed").glob("*.csv")))

Inbox: []
Rejected: []
Processed: [WindowsPath('data/processed/Lagos_weather.csv')]


In [353]:
# Let's remove the test records from the database
# This clears the test files’ rows from the log and from staging. Skip it if you’d rather keep them as a record of your tests.
# So we skipped this process.


In [354]:
# Next phase: scheduling

# Our pipeline now works end to end, so the remaining job is making it run by itself every day. 
# This is the cron part of our original plan, and on Windows the equivalent is Task Scheduler.
# Task Scheduler is best at launching a .bat file, a small script that holds the command you’ve been typing by hand. We’ll do it in three steps:
# 1. Create the .bat file (now)
# 2. Test it by double-clicking
# 3. Create the Task Scheduler task to run it daily

In [355]:
# Now after all, there is a missing piece, this pipeline runs and reads a new file when i manually download the weather csv file from Open Mateo API
# So we need an automated download script to get the files automatically from their API, which we have created on VScode.

%load_ext autoreload
%autoreload 2

from src.download import download_lagos_csv

new_file = download_lagos_csv()
new_file

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


WindowsPath('data/inbox/Lagos_weather_2026-10-18.csv')

In [356]:
# Let's Inspect the new file
from src.extract import read_weather_csv

test_new = read_weather_csv(new_file)
test_new.head()

,time,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm)
0,2026-09-18,28.7,24.3,13.5
1,2026-09-19,28.7,25.4,15.6
2,2026-09-20,27.6,24.5,7.5
3,2026-09-21,28.2,25.4,11.7
4,2026-09-22,27.5,25.0,16.7


In [357]:
# Let's dry run the Validation of the data to see if pass validation checks
from src.validate import validate_weather

results = validate_weather(test_new)
{name: ok for name, ok in results.items() if not ok}

{}

In [358]:
# Let's check our resultd after adding and validatin our download.py script for new file into our pipeline
pd.read_sql("SELECT COUNT(*) AS total_rows, MIN(date) AS first_day, MAX(date) AS last_day FROM weather_daily", engine)

,total_rows,first_day,last_day
0,1858,2020-01-01,2026-10-18


In [359]:
# the output above shows, 31 extra rows has been added to the transformed data (weather daily)

In [360]:
# Let's check the log again
pd.read_sql("SELECT run_id, file_name, rows_loaded, status FROM etl_run_log ORDER BY run_id", engine)

,run_id,file_name,rows_loaded,status
0,1,Lagos_weather.csv,1827,success
1,5,Lagos_weather_test.csv,1827,success
2,7,Lagos_broken.csv,0,rejected
3,8,Lagos_weather_2026-10-18.csv,31,success


In [361]:
# Next phase: pytest tests
# Tests are short checks that tell you immediately if a code change breaks something. 
# We’ll start with transform_weather, because it’s the easiest to test.

In [362]:
%pip install pytest

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///lag_weather.db")

pd.read_sql("""
    SELECT 'weather_daily' AS tbl, MAX(date) AS latest, COUNT(*) AS n FROM weather_daily
    UNION ALL
    SELECT 'stg_weather', MAX(time), COUNT(*) FROM stg_weather
""", engine)

,tbl,latest,n
0,weather_daily,2026-10-28,1868
1,stg_weather,2026-10-28,1889


In [4]:
# There is a gap in the the dates at some point from 2024 - 2026, so we want to fill up the gap.
from src.download import download_lagos_csv
download_lagos_csv(days_back=700)

WindowsPath('data/inbox/Lagos_weather_2026-10-28.csv')

In [6]:
# Now let's run the pipeline in our VScode below

# & C:\\ProgramData\\anaconda3\\python.exe run_pipeline.py

In [7]:
# Let's check if the gap is filled

pd.read_sql("""
    SELECT COUNT(*) AS n,
           MIN(date) AS first_day,
           MAX(date) AS last_day,
           CAST(julianday(MAX(date)) - julianday(MIN(date)) + 1 AS INTEGER) AS expected_days
    FROM weather_daily
""", engine)

,n,first_day,last_day,expected_days
0,1868,2020-01-01,2026-10-28,2493


In [9]:
# It did not work, so let's check where the file is
from pathlib import Path

print("Inbox:    ", [p.name for p in Path("data/inbox").glob("*.csv")])
print("Rejected: ", [p.name for p in Path("data/rejected").glob("*.csv")])
print("Processed:", sorted(p.name for p in Path("data/processed").glob("*.csv")))

Inbox:     ['Lagos_weather_2026-10-18.csv', 'Lagos_weather_2026-10-28.csv']
Rejected:  []
Processed: ['Lagos_weather.csv', 'Lagos_weather_2026-10-18.csv', 'Lagos_weather_2026-10-28.csv']


In [10]:
pd.read_sql("""
    SELECT run_id, file_name, rows_read, rows_loaded, status, error_message, run_at
    FROM etl_run_log
    ORDER BY run_id DESC
    LIMIT 5
""", engine)

,run_id,file_name,rows_read,rows_loaded,status,error_message,run_at
0,9,Lagos_weather_2026-10-28.csv,31,31,success,None,2026-11-02T08:01:04.833723
1,8,Lagos_weather_2026-10-18.csv,31,31,success,None,2026-10-23T13:48:00.308145
2,7,Lagos_broken.csv,1827,0,rejected,Failed checks: temp_max_range,2026-10-23T11:55:05.507825
3,5,Lagos_weather_test.csv,1827,1827,success,None,2026-10-23T11:02:30.771424
4,1,Lagos_weather.csv,1827,1827,success,None,2026-10-19T16:46:05.695236


In [13]:
# So let's understand what will happen niw in the next blocks of code.

# Our database was missing about 21 months of weather (early 2025 to September 2026), 
# so we downloaded one big file to cover it. The first attempt failed because of a **name clash**: 
# the big file and the pipeline's own daily download were both saved as `Lagos_weather_2026-10-28.csv`, 
# so the daily download overwrote the big file before the pipeline could read it. 
# Renaming the big file to `Lagos_weather_backfill.csv` gave it a unique name, so the pipeline saw it as new, validated it and loaded it into `weather_daily`.
# The upsert filled the gap without duplicating existing days, leaving one row for every day from 2020 to today (2,493 rows).

import pandas as pd
from src.download import download_lagos_csv

f = download_lagos_csv(days_back=700)
backfill = f.with_name("Lagos_weather_backfill.csv")
f.replace(backfill)

print(backfill)
print(len(pd.read_csv(backfill, skiprows=3)), "rows")

data\inbox\Lagos_weather_backfill.csv
701 rows


In [12]:
# Let's conform if it works.
pd.read_sql("""
    SELECT COUNT(*) AS n,
           MIN(date) AS first_day,
           MAX(date) AS last_day,
           CAST(julianday(MAX(date)) - julianday(MIN(date)) + 1 AS INTEGER) AS expected_days
    FROM weather_daily
""", engine)

,n,first_day,last_day,expected_days
0,2493,2020-01-01,2026-10-28,2493


In [14]:
# Now we are moving to test the Validate_weather to know if our validations works well.
# What it does is that, it checks a file to see if it's good or bad and allows the good one and stops the bad one.

import pandas as pd
from src.validate import validate_weather


def make_good():
    return pd.DataFrame({
        "time": ["2024-01-01", "2024-01-02", "2024-01-03"],
        "temperature_2m_max (°C)": [33.0, 32.0, 31.0],
        "temperature_2m_min (°C)": [22.0, 21.0, 23.0],
        "precipitation_sum (mm)": [0.0, 1.2, 0.0],
    })


def test_good_data_passes_every_check():
    results = validate_weather(make_good())
    assert all(results.values())


def test_null_value_fails_no_nulls():
    df = make_good()
    df.loc[1, "precipitation_sum (mm)"] = None
    assert validate_weather(df)["no_nulls"] is False


def test_missing_day_fails_no_missing_days():
    df = make_good().drop(index=1)
    assert validate_weather(df)["no_missing_days"] is False


def test_duplicate_date_fails_unique_dates():
    df = pd.concat([make_good(), make_good().iloc[[0]]], ignore_index=True)
    assert validate_weather(df)["unique_dates"] is False


def test_impossible_temperature_fails_range():
    df = make_good()
    df.loc[0, "temperature_2m_max (°C)"] = 99.0
    assert validate_weather(df)["temp_max_range"] is False


def test_max_below_min_fails_consistency():
    df = make_good()
    df.loc[0, "temperature_2m_max (°C)"] = 20.0
    df.loc[0, "temperature_2m_min (°C)"] = 25.0
    assert validate_weather(df)["max_gte_min"] is False


def test_future_date_fails_timeliness():
    df = make_good()
    df.loc[2, "time"] = "2099-01-03"
    assert validate_weather(df)["no_future_dates"] is False

In [ ]:
# Now let's run this above in VScode using this below
# & C:\\ProgramData\\anaconda3\\python.exe -m pytest